# 07 - Benchmark: Delta vs Iceberg vs Hudi

This notebook compares three major data lake table formats:

- Delta Lake
- Apache Iceberg
- Apache Hudi

You will benchmark:

- Write performance
- Read performance
- Upsert / merge performance
- File layout & metadata behavior

This is a simplified benchmark suitable for a local Spark cluster.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col

spark = (
    SparkSession.builder
    .appName("Benchmark-Delta-Iceberg-Hudi")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

In [2]:
BASE_PATH="/workspace/data/benchmark"

DELTA_PATH=f"{BASE_PATH}/delta"
ICEBERG_TABLE="local.db.iceberg_bench"
HUDI_PATH=f"{BASE_PATH}/hudi"

## Generate test dataset

In [3]:
df = spark.range(0, 100000).withColumn("value", rand()).withColumn("category", (col("id") % 10).cast("string"))

df.cache()
df.count()

100000

## Delta write

In [4]:
import time
start=time.time()

df.write.format("delta").mode("overwrite").save(DELTA_PATH)

print("Delta write:", time.time()-start)

Delta write: 5.403834104537964


## Iceberg write

In [6]:
ICEBERG_NAMESPACE = "local.db"
ICEBERG_TABLE = f"{ICEBERG_NAMESPACE}.iceberg_bench"

spark.sql("CREATE NAMESPACE IF NOT EXISTS local.db")

start = time.time()

df.writeTo(ICEBERG_TABLE) \
  .using("iceberg") \
  .createOrReplace()

print("Iceberg write:", time.time() - start)

Iceberg write: 2.267979145050049


## Hudi write

In [7]:
start=time.time()

df.write.format("hudi")
            .option("hoodie.table.name","hudi_bench")
            .option("hoodie.datasource.write.recordkey.field","id")
            .option("hoodie.datasource.write.precombine.field","id")
            .option("hoodie.datasource.write.table.type","COPY_ON_WRITE")
            .mode("overwrite")
            .save(HUDI_PATH)

print("Hudi write:", time.time()-start)

Hudi write: 18.873035192489624


## Read benchmark

In [8]:
start=time.time()
spark.read.format("delta").load(DELTA_PATH).count()
print("Delta read:", time.time()-start)

start=time.time()
spark.read.table(ICEBERG_TABLE).count()
print("Iceberg read:", time.time()-start)

start=time.time()
spark.read.format("hudi").load(HUDI_PATH).count()
print("Hudi read:", time.time()-start)

26/04/25 20:20:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Delta read: 3.552523612976074
Iceberg read: 0.37827539443969727
# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf
# WARNING: Unable to attach Serviceability Agent. You can try again with escalated privileges. Two options: a) use -Djol.tryWithSudo=true to try with sudo; b) echo 0 | sudo tee /proc/sys/kernel/yama/ptrace_scope
Hudi read: 2.9054179191589355


## Summary

Typical observations:

- Delta: fastest writes, strong ecosystem
- Iceberg: best metadata & evolution
- Hudi: best for upserts & CDC

Actual performance depends on workload.